In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import numpy as np

In [7]:

# Knowledge base
docs = [
    "AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes.",
    "The system can use satellite data, weather data, water temperature, and sensor measurements.",
    "Dense retrieval uses embeddings to find documents with similar meaning.",
    "BM25 is a sparse retrieval method based on keyword matching.",
    "Transformers are neural network architectures used in models such as GPT, BERT, and LLaMA."
]

query = "How does AI4HAB predict algae blooms?"

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embedding_model.encode(docs)
query_embedding = embedding_model.encode([query])

# Similarity search
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_k = 2
top_indices = np.argsort(similarities)[::-1][:top_k]
retrieved_docs = [docs[i] for i in top_indices]

print("Retrieved Documents:\n")
for doc in retrieved_docs:
    print("-", doc)


/Users/swatichandna/.pyenv/versions/3.11.7/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Retrieved Documents:

- AI4HAB uses artificial intelligence to forecast harmful algal blooms in lakes.
- The system can use satellite data, weather data, water temperature, and sensor measurements.


In [8]:

# Build context
context = " ".join(retrieved_docs)

# This is what LLM will see as input
# Improved prompt
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Write one clear sentence.
Answer:
"""

# Better text generation model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

answer = generator(
    prompt,
    max_new_tokens=80,
    do_sample=False 
)[0]["generated_text"]

print("\nGenerated Answer:\n")
print(answer)


Generated Answer:

The system can use satellite data, weather data, water temperature, and sensor measurements
